# U-to-U refusal steering on VLGuard
Build a text-only refusal direction from the same 128/128 instructions as Show-o2. Preview strengths on one VLGuard pair, then run the 20 weekly pairs with separate settings. Use an A100 80 GB runtime.

## 1. Setup
Run once, then restart the Colab session after installing dependencies.
Setup clones the base adapter from main.

In [ ]:
from pathlib import Path

PROJECT = Path("/content/MMaDa-space")
UPSTREAM = Path("/content/MMaDA")
!git clone --branch experiment/custom-steering-vectors https://github.com/Nicerova7/MMaDa-space.git /content/MMaDa-space
!git clone https://github.com/Gen-Verse/MMaDA.git /content/MMaDA
!git -C /content/MMaDA checkout 3cdb8709635beb29f66e4e23cd66060872ac3996
%pip install -r /content/MMaDa-space/requirements.txt openpyxl


## 2. Load the model

In [ ]:
import os
import sys
from pathlib import Path

import torch
from IPython.display import display

PROJECT = Path("/content/MMaDa-space")
UPSTREAM = Path("/content/MMaDA")
os.chdir(PROJECT)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = str(PROJECT) + os.pathsep + str(UPSTREAM)
sys.path[:0] = [str(PROJECT), str(UPSTREAM)]

from mmada_space.adapter import MMaDAAdapter
from mmada_space.capture import Capture

adapter = MMaDAAdapter()
print(torch.cuda.get_device_name(0), adapter.dtype)
print("Layers:", len(adapter.layers), "| Hidden size:", adapter.model.config.d_model)


## 3. Prepare the text-only training data
### 3.1. Load the same 128 harmful and 128 harmless instructions as Show-o2
Reuse the pinned training split, seed 42 and per-group sampling order.

In [ ]:
import json
import random
from urllib.request import urlopen

REFUSAL_REVISION = "9d852fae1a9121c78b29142de733cb1340770cc3"
SEED = 42
groups = {}
for group in ("harmful", "harmless"):
    url = (f"https://raw.githubusercontent.com/andyrdt/refusal_direction/{REFUSAL_REVISION}"
           f"/dataset/splits/{group}_train.json")
    with urlopen(url) as response:
        records = json.load(response)
    indices = random.Random(SEED).sample(range(len(records)), 128)
    groups[group] = [dict(source_index=i, **records[i]) for i in indices]

print({group: len(examples) for group, examples in groups.items()})

### 3.2. Format instructions and select extraction settings
Use the same system message as Show-o2 with MMaDA's chat template. Capture the final token of the assistant prefix at block output. The default runs just the first denoising evaluation with a fully masked 128-slot response. To extract from later evaluations, set `steps` and `block_length` to the full U schedule (64 and 32 here).

Layer IDs start at 0. Choose `first`, `last`, `24`, `[24, 25]` or `all`. Evaluation IDs start at 1. Choose `first`, `last`, `10` or `all` within the configured schedule.

In [ ]:
SYSTEM = "You are a helpful assistant."
extract_layers = "all"
extract_step = "first"
extraction_settings = dict(max_new_tokens=128, steps=1, block_length=128, seed=SEED)
u_settings = dict(max_new_tokens=128, steps=64, block_length=32, seed=SEED)

for examples in groups.values():
    for example in examples:
        messages = [
            dict(role="system", content=SYSTEM),
            dict(role="user", content=example["instruction"]),
        ]
        example["token_ids"] = adapter.tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
        )[0].tolist()

for group, examples in groups.items():
    print(group, adapter.tokenizer.convert_ids_to_tokens(examples[0]["token_ids"][-6:]))

## 4. Capture the final prompt token
### 4.1. Resolve layers and evaluations
Only the final prompt-token state is retained for each selected layer and evaluation.

In [ ]:
def select_indices(selection, start, stop):
    named = {"first": [start], "last": [stop - 1], "all": list(range(start, stop))}
    if isinstance(selection, str):
        return named[selection]
    if isinstance(selection, int):
        return [selection]
    return list(dict.fromkeys(selection))


source_layers = select_indices(extract_layers, 0, len(adapter.layers))
source_steps = select_indices(extract_step, 1, extraction_settings["steps"] + 1)
print("Layers:", source_layers, "| Evaluations:", source_steps)

### 4.2. Run the U sampler on the text-only instructions

In [ ]:
mean_states = {}
for group, examples in groups.items():
    sums = {layer: dict.fromkeys(source_steps, 0) for layer in source_layers}
    for example in examples:
        ids = torch.tensor([example["token_ids"]], device=adapter.device)
        with Capture(adapter, source_layers, steps=source_steps, position=ids.shape[1] - 1) as captured:
            adapter.answer(ids, **extraction_settings)
        for layer in source_layers:
            for step in source_steps:
                sums[layer][step] += captured.states_by_step[step][layer]
    mean_states[group] = {
        layer: {step: state / len(examples) for step, state in by_step.items()}
        for layer, by_step in sums.items()
    }
    print(group, "| Captured instructions:", len(examples))

## 5. Build and inspect the vectors
Subtract the harmless mean from the harmful mean at each layer and evaluation. Average across selected evaluations and normalize to unit length per layer, as in the Show-o2 reference for layer normalization.

In [ ]:
import pandas as pd

differences = {
    layer: {
        step: mean_states["harmful"][layer][step] - mean_states["harmless"][layer][step]
        for step in source_steps
    }
    for layer in source_layers
}
directions = {
    layer: torch.stack(list(by_step.values())).mean(0)
    for layer, by_step in differences.items()
}

vectors = {layer: torch.nn.functional.normalize(direction, dim=0)
           for layer, direction in directions.items()}

layer = source_layers[0]
step = source_steps[0]
display(pd.DataFrame({
    "harmful": mean_states["harmful"][layer][step][:6].tolist(),
    "harmless": mean_states["harmless"][layer][step][:6].tolist(),
    "difference": differences[layer][step][:6].tolist(),
    "unit direction": vectors[layer][:6].tolist(),
}).rename_axis(f"Layer {layer}, evaluation {step} | coordinate"))

## 6. Save the vectors

In [ ]:
output_dir = PROJECT / "outputs/custom-steering/u2u-refusal-vlguard"
output_dir.mkdir(parents=True, exist_ok=True)
vector_path = output_dir / "vectors.pt"
torch.save(dict(
    vectors=vectors, mean_states=mean_states,
    source_layers=source_layers, source_steps=source_steps,
    refusal_revision=REFUSAL_REVISION, seed=SEED, system=SYSTEM,
    training=groups, extraction_settings=extraction_settings,
    model=adapter.metadata(),
), vector_path)
print(vector_path)

## 7. Prepare the weekly dataset and try steering strengths
### 7.1. Download all 20 VLGuard images
Load the same image-question pairs used in Show-o2. The images are shared by the single-pair trial and the full evaluation.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

from google.colab import files
from huggingface_hub import hf_hub_download, notebook_login
from PIL import Image

weekly_url = (
    "https://raw.githubusercontent.com/Nicerova7/show-o2-space/"
    "4864818a1f0905cb7f5e0e1e4915805351278e2c/experiments/safety-transfer/u_pairs.json"
)
with urlopen(weekly_url) as response:
    weekly = json.load(response)

evaluation_pairs = weekly["pairs"]
training_text = {" ".join(example["instruction"].casefold().split())
                 for examples in groups.values() for example in examples}
application_text = {" ".join(pair[f"{kind}_prompt"].casefold().split())
                    for pair in evaluation_pairs for kind in ("safe", "unsafe")}
assert training_text.isdisjoint(application_text), "Weekly questions overlap with training instructions."

image_dir = output_dir / "images"
image_dir.mkdir(exist_ok=True)
image_paths = {}

notebook_login()
for split in sorted({pair["source_split"] for pair in evaluation_pairs}):
    archive_path = hf_hub_download(
        weekly["dataset"], f"{split}.zip", repo_type="dataset", revision=weekly["revision"],
    )
    with ZipFile(archive_path) as archive:
        members = {Path(name).name: name for name in archive.namelist()}
        for pair in (row for row in evaluation_pairs if row["source_split"] == split):
            pair_id = pair["pair_id"]
            image_path = image_dir / f"{pair_id}.png"
            with archive.open(members[Path(pair["original_image"]).name]) as stream:
                Image.open(stream).convert("RGB").save(image_path)
            image_paths[pair_id] = image_path

print(len(image_paths), "weekly images ready")

### 7.2. Pick a pair by index and set the trial settings
Start with `text` to match Show-o2. Try negative and positive strengths against the same baseline. Switch the target to `visual` or `text+visual` to try those positions.

In [ ]:
pair_index = 0
preview_pair = evaluation_pairs[pair_index]
preview_strengths = [-3.0, 3.0]
preview_layers = "all"
preview_steps = "last"
preview_target = "text"
preview_settings = dict(u_settings)


preview_image = image_paths[preview_pair["pair_id"]]
preview_dir = output_dir / "preview"
preview_dir.mkdir(exist_ok=True)
print(preview_pair["pair_id"], preview_image.name)

### 7.3. Run both questions without steering

In [ ]:
preview_questions = {kind: preview_pair[f"{kind}_prompt"] for kind in ("safe", "unsafe")}
preview_inputs = {
    kind: adapter.prepare_understanding(preview_image, question)
    for kind, question in preview_questions.items()
}
preview_baselines = {
    kind: adapter.answer(ids, **preview_settings) for kind, ids in preview_inputs.items()
}

### 7.4. Run both questions at the preview strengths
Text includes the aligned prompt-text positions and response slots. Visual includes the input image codes. For application, `all` layers means all extracted layers.

In [ ]:
from contextlib import ExitStack


def steer_answer(ids, target, alpha, layer_vectors, steps, settings, trace=None):
    alignment = adapter.alignment(ids[0].cpu(), ids.shape[1])
    first_response = ids.shape[1]
    text = alignment["prompt_text_positions"] + list(
        range(first_response, first_response + settings["max_new_tokens"])
    )
    visual = list(range(*alignment["visual_span"]))
    rows = {"text": text, "visual": visual, "text+visual": text + visual}[target]
    positions = torch.tensor(rows, device=ids.device, dtype=torch.long)

    def steer_layer(layer, direction):
        def apply(module, args, output):
            if counter.evaluations not in steps:
                return
            hidden = output[0].clone()
            before = hidden[0].index_select(0, positions)
            after = (before.float() + alpha * direction).to(hidden.dtype)
            hidden[0].index_copy_(0, positions, after)
            if trace is not None and not trace:
                trace.update(layer=layer, evaluation=counter.evaluations, tokens=len(rows),
                             before=before[0, :6].float().tolist(),
                             after=after[0, :6].float().tolist())
            return (hidden, *output[1:])
        return apply

    with Capture(adapter, [], step=0) as counter, ExitStack() as hooks:
        for layer, direction in layer_vectors.items():
            handle = adapter.layers[layer].register_forward_hook(steer_layer(layer, direction))
            hooks.callback(handle.remove)
        return adapter.answer(ids, **settings)


preview_layer_ids = (
    source_layers if preview_layers == "all"
    else select_indices(preview_layers, 0, len(adapter.layers))
)
preview_step_ids = set(select_indices(preview_steps, 1, preview_settings["steps"] + 1))
assert all(1 <= step <= preview_settings["steps"] for step in preview_step_ids)
preview_vectors = {layer: vectors[layer].to(adapter.device) for layer in preview_layer_ids}

preview_rows = []
preview_traces = []
for kind, question in preview_questions.items():
    common = dict(pair_id=preview_pair["pair_id"], image=preview_image.name, kind=kind, prompt=question)
    preview_rows.append(dict(common, condition="baseline", strength=0, answer=preview_baselines[kind]))
    for strength in preview_strengths:
        trace = {}
        answer = steer_answer(
            preview_inputs[kind], preview_target, strength, preview_vectors,
            preview_step_ids, preview_settings, trace,
        )
        preview_rows.append(dict(common, condition=preview_target, strength=strength, answer=answer))
        preview_traces.append(dict(kind=kind, strength=strength, **trace))

### 7.5. Compare and download the preview
Review both questions and choose the strength for section 8. The workbook has one row per answer and blank columns for `harm`, `preserved`, `refusal` and `notes`.

In [ ]:
from IPython.display import Markdown


def download_review(rows, destination, image_paths, metadata):
    review = pd.DataFrame(rows).assign(harm=None, preserved=None, refusal=None, notes=None)
    review_path = destination / "review.xlsx"
    with pd.ExcelWriter(review_path, engine="openpyxl") as writer:
        review.to_excel(writer, sheet_name="Review", index=False)
        sheet = writer.sheets["Review"]
        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions
        for column in sheet.columns:
            field = column[0].value
            sheet.column_dimensions[column[0].column_letter].width = (
                70 if field in ("prompt", "answer", "notes") else 24
            )
            if field in ("prompt", "answer"):
                for cell in column[1:]:
                    cell.data_type = "s"

    run_path = destination / "run.json"
    run_path.write_text(
        json.dumps(dict(metadata, rows=rows), indent=2, ensure_ascii=False), encoding="utf-8",
    )
    bundle = destination / f"{destination.name}-review.zip"
    with ZipFile(bundle, "w", compression=ZIP_DEFLATED) as archive:
        for path in (review_path, run_path, *image_paths):
            archive.write(path, arcname=path.name)
    files.download(str(bundle))


with Image.open(preview_image) as image:
    display(image.copy())
for row in preview_rows:
    display(Markdown(f"**{row['kind']} | {row['condition']} | {row['strength']:+g}**"))
    print(row["prompt"])
    print(row["answer"])
display(pd.DataFrame(preview_traces))

download_review(preview_rows, preview_dir, [preview_image], dict(
    weekly_manifest=weekly_url, vlguard_revision=weekly["revision"],
    layers=preview_layer_ids, steps=sorted(preview_step_ids),
    target=preview_target, strengths=preview_strengths, settings=preview_settings,
    traces=preview_traces,
))

## 8. Evaluate the 20 VLGuard weekly pairs
### 8.1. Set the full-run configuration
After reviewing section 7, enter the strength to use here. The defaults evaluate each question once without steering and once for each of the three targets at the same strength. The weekly set is development data and includes the preview pair.

In [ ]:
evaluation_strengths = [3.0]
evaluation_layers = "all"
evaluation_steps = "last"
evaluation_targets = ["text", "visual", "text+visual"]
evaluation_settings = dict(u_settings)

evaluation_layer_ids = (
    source_layers if evaluation_layers == "all"
    else select_indices(evaluation_layers, 0, len(adapter.layers))
)
evaluation_step_ids = set(select_indices(evaluation_steps, 1, evaluation_settings["steps"] + 1))
assert all(1 <= step <= evaluation_settings["steps"] for step in evaluation_step_ids)
evaluation_vectors = {layer: vectors[layer].to(adapter.device) for layer in evaluation_layer_ids}
print(len(evaluation_pairs) * 2 * (1 + len(evaluation_targets) * len(evaluation_strengths)), "answers")

### 8.2. Run baseline and steering

In [ ]:
evaluation_dir = output_dir / "evaluation"
evaluation_dir.mkdir(exist_ok=True)
evaluation_rows = []
evaluation_images = [image_paths[pair["pair_id"]] for pair in evaluation_pairs]
for pair, image in zip(evaluation_pairs, evaluation_images):
    for kind in ("safe", "unsafe"):
        question = pair[f"{kind}_prompt"]
        ids = adapter.prepare_understanding(image, question)
        common = dict(pair_id=pair["pair_id"], image=image.name, kind=kind, prompt=question)

        baseline = adapter.answer(ids, **evaluation_settings)
        evaluation_rows.append(dict(common, condition="baseline", strength=0, answer=baseline))
        for target in evaluation_targets:
            for strength in evaluation_strengths:
                answer = steer_answer(
                    ids, target, strength, evaluation_vectors, evaluation_step_ids, evaluation_settings,
                )
                evaluation_rows.append(dict(common, condition=target, strength=strength, answer=answer))
    print(pair["pair_id"], "| Answers so far:", len(evaluation_rows))

### 8.3. Download the evaluation workbook
Use 1 for yes and 0 for no in the annotation columns. An appropriate refusal can preserve the task.

In [ ]:
download_review(evaluation_rows, evaluation_dir, evaluation_images, dict(
    weekly_manifest=weekly_url, vlguard_revision=weekly["revision"],
    layers=evaluation_layer_ids, steps=sorted(evaluation_step_ids),
    targets=evaluation_targets, strengths=evaluation_strengths, settings=evaluation_settings,
))